# Importy i stałe

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [24]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader, Dataset
import os
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import json
import re
from tqdm.notebook import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Używam urządzenia: {device}")

PROJECT_DIR = '/content/drive/Shareddrives/adv_images/adversarial_images/'
LABELS_CLEAN_VGG = PROJECT_DIR + 'prediction_results/clean_VGG.json'

FEATURE_EXTRACTION_RESULTS = PROJECT_DIR + "feature_extraction/"

Używam urządzenia: cuda


# Definicje klas
- AdversarialImageDataset
- AttackMetadata
- ImageInfo
- InceptionModule
- InceptionCIFAR

In [9]:
class AdversarialImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.image_files = []

        if os.path.exists(root_dir):
            for file in os.listdir(root_dir):
                if file.endswith(".png"):
                    self.image_files.append(file)

        # Sortowanie po indeksie z nazwy (adv_image_0_label_3.png -> 0)
        try:
            self.image_files.sort(key=lambda x: int(x.split('_')[2]))
        except Exception as e:
            print("Błąd sortowania. Sprawdź nazwy plików.")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        path = os.path.join(self.root_dir, self.image_files[idx])
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        # Zwracamy sam obraz do modelu
        return image, self.image_files[idx]

In [10]:
class AttackMetadata:
    def __init__(self, attack_name: str, generated_by: str, model_attacked: str):
        self.attack_name = attack_name
        self.generated_by = generated_by
        self.model_attacked = model_attacked

    def to_dict(self) -> dict:
        return {"attack_name": self.attack_name, "generated_by": self.generated_by, "model_attacked": self.model_attacked}

In [11]:
class ImageInfo:
    def __init__(
        self,
        attack_metadata: AttackMetadata,
        image_index: int,
        true_label: int,
        clean_prediction_label: int,
        adversarial_prediction_label: int,
        clean_feature_map=None,
        adversarial_feature_map=None
    ):
        self.image_index = image_index
        self.true_label = true_label

        self.clean_prediction_label = clean_prediction_label
        self.adversarial_prediction_label = adversarial_prediction_label

        self.clean_feature_map = clean_feature_map
        self.adversarial_feature_map = adversarial_feature_map

        self.attack_metadata = attack_metadata

    def is_clean_prediction_correct(self) -> bool:
        return self.clean_prediction_label == self.true_label

    def is_adversarial_prediction_correct(self) -> bool:
        return self.adversarial_prediction_label == self.true_label

    def was_attack_successful(self) -> bool:
        return (self.clean_prediction_label == self.true_label and self.adversarial_prediction_label != self.true_label)

    def __repr__(self) -> str:
        return (
            f"ImageInfo("
            f"image_index={self.image_index}, "
            f"true_label={self.true_label}, "
            f"clean_prediction_label={self.clean_prediction_label}, "
            f"adversarial_prediction_label={self.adversarial_prediction_label}, "
            f"attack_name={self.attack_metadata.attack_name}, "
            f"clean_feature_map_set={self.clean_feature_map is not None}, "
            f"adversarial_feature_map_set={self.adversarial_feature_map is not None}"
            f")"
        )

    def to_dict(self) -> dict:
        data = {
            "image_index": self.image_index,
            "true_label": self.true_label,

            "clean_prediction_label": self.clean_prediction_label,
            "adversarial_prediction_label": self.adversarial_prediction_label,

            "is_clean_prediction_correct": self.is_clean_prediction_correct(),
            "is_adversarial_prediction_correct": self.is_adversarial_prediction_correct(),
            "was_attack_successful": self.was_attack_successful(),
            "attack_metadata": self.attack_metadata.to_dict()
        }
        # data.update(self.get_feature_map_shapes())
        return data


In [12]:
class InceptionModule(nn.Module):
    def __init__(self, in_channels,n1x1, n3x3red, n3x3, n5x5red, n5x5, pool_proj):
        super().__init__()

        # 1x1 conv
        self.b1 = nn.Sequential(
            nn.Conv2d(in_channels, n1x1, kernel_size=1, bias=False),
            nn.BatchNorm2d(n1x1),
            nn.ReLU(inplace=True)
        )

        # 1x1 -> 3x3 conv
        self.b2 = nn.Sequential(
            nn.Conv2d(in_channels, n3x3red, kernel_size=1, bias=False),
            nn.BatchNorm2d(n3x3red),
            nn.ReLU(inplace=True),
            nn.Conv2d(n3x3red, n3x3, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(n3x3),
            nn.ReLU(inplace=True)
        )

        # 1x1 -> 5x5 conv
        self.b3 = nn.Sequential(
            nn.Conv2d(in_channels, n5x5red, kernel_size=1, bias=False),
            nn.BatchNorm2d(n5x5red),
            nn.ReLU(inplace=True),
            nn.Conv2d(n5x5red, n5x5, kernel_size=5, padding=2, bias=False),
            nn.BatchNorm2d(n5x5),
            nn.ReLU(inplace=True)
        )

        # 3x3 pool -> 1x1 conv
        self.b4 = nn.Sequential(
            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),
            nn.Conv2d(in_channels, pool_proj, kernel_size=1, bias=False),
            nn.BatchNorm2d(pool_proj),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return torch.cat([self.b1(x), self.b2(x), self.b3(x), self.b4(x)], dim=1)

In [13]:
class InceptionCIFAR(nn.Module):
    def __init__(self, num_classes=10, dropout=0.3):
        super().__init__()

        # 7x7 stride=2 -> 3x3 stride=1
        self.stem = nn.Sequential(
            nn.Conv2d(3, 96, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True)
        )

        # ---- Inception blocks: 32x32 ----
        self.inception3a = InceptionModule(96, 32, 48, 64, 8, 16, 16)
        self.inception3b = InceptionModule(128, 64, 64, 96, 16, 32, 32)
        self.maxpool1 = nn.MaxPool2d(3, stride=2, padding=1)  # 32x32 -> 16x16

        # ---- 16x16 ----
        self.inception4a = InceptionModule(224, 96, 48, 104, 8, 24, 32)
        self.inception4b = InceptionModule(256, 80, 64, 128, 16, 32, 32)
        self.maxpool2 = nn.MaxPool2d(3, stride=2, padding=1)  # 16x16 -> 8x8

        # ---- 8x8 ----
        self.inception5a = InceptionModule(272, 128, 80, 160, 16, 48, 48)
        self.inception5b = InceptionModule(384, 192, 96, 192, 24, 64, 64)

        # 7x7 avg pool -> adaptive 1x1
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(512, num_classes)

    def forward_features(self, x):
        x = self.stem(x)

        x = self.inception3a(x)
        x = self.inception3b(x)
        x = self.maxpool1(x)

        x = self.inception4a(x)
        x = self.inception4b(x)
        x = self.maxpool2(x)

        x = self.inception5a(x)
        x = self.inception5b(x)

        x = self.avgpool(x)
        x = torch.flatten(x, 1)

        return x # (B, 512)

    def forward(self, x):
        features = self.forward_features(x)
        features = self.dropout(features)
        logits = self.fc(features)
        return logits


# Przygotowanie informacji o obrazach
dla ataków wygenerowanych na resnet (One Pixel, PGD, FGSM, Deep Fool, CW, Auto Attack)

In [14]:
mean, std = [0.4914, 0.4822, 0.4465], [0.247, 0.2435, 0.2616]
BATCH_SIZE = 128

preprocess = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

clean_dataset = CIFAR10(root='./data', train=False, download=True, transform=preprocess)
clean_loader = DataLoader(clean_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
print(f"Dane gotowe. Clean: {len(clean_dataset)}")

100%|██████████| 170M/170M [00:04<00:00, 35.1MB/s]


Dane gotowe. Clean: 10000


In [15]:
# model for attack genration | attack name | model being attacked:

# Resnet | OnePixel | VGG
LABELS_RESNET_ONEPIXEL_VGG = PROJECT_DIR + 'prediction_results/ResNet_OnePixel_VGG.json'
IMAGES_RESNET_ONEPIXEL_VGG_DIR = PROJECT_DIR + 'test_ResNet_OnePixel'

# NAMING: adv_{model_attack_genration}_{attack_name}_{model_being_attacked}_{object_name}
print(IMAGES_RESNET_ONEPIXEL_VGG_DIR)
adv_resnet_onepixel_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_ONEPIXEL_VGG_DIR, transform=preprocess)
adv_resnet_onepixel_vgg_loader = DataLoader(adv_resnet_onepixel_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | PGD | VGG
LABELS_RESNET_PGD_VGG = PROJECT_DIR + 'prediction_results/ResNet_PGD_VGG.json'
IMAGES_RESNET_PGD_VGG_DIR = PROJECT_DIR + 'test_ResNet_PGD'

print(IMAGES_RESNET_PGD_VGG_DIR)
adv_resnet_pgd_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_PGD_VGG_DIR, transform=preprocess)
adv_resnet_pgd_vgg_loader = DataLoader(adv_resnet_pgd_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | FGSM | VGG
LABELS_RESNET_FGSM_VGG = PROJECT_DIR + 'prediction_results/ResNet_FGSM_VGG.json'
IMAGES_RESNET_FGSM_VGG_DIR = PROJECT_DIR + 'test_ResNet_FGSM'

print(IMAGES_RESNET_FGSM_VGG_DIR)
adv_resnet_fgsm_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_FGSM_VGG_DIR, transform=preprocess)
adv_resnet_fgsm_vgg_loader = DataLoader(adv_resnet_fgsm_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | DeepFool | VGG
LABELS_RESNET_DEEPFOOL_VGG = PROJECT_DIR + 'prediction_results/ResNet_DeepFool_VGG.json'
IMAGES_RESNET_DEEPFOOL_VGG_DIR = PROJECT_DIR + 'test_ResNet_DeepFool'

print(IMAGES_RESNET_DEEPFOOL_VGG_DIR)
adv_resnet_deepfool_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_DEEPFOOL_VGG_DIR, transform=preprocess)
adv_resnet_deepfool_vgg_loader = DataLoader(adv_resnet_deepfool_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | CW | VGG
LABELS_RESNET_CW_VGG = PROJECT_DIR + 'prediction_results/ResNet_CW_VGG.json'
IMAGES_RESNET_CW_VGG_DIR = PROJECT_DIR + 'test_ResNet_CW'

print(IMAGES_RESNET_CW_VGG_DIR)
adv_resnet_cw_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_CW_VGG_DIR, transform=preprocess)
adv_resnet_cw_vgg_loader = DataLoader(adv_resnet_cw_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Resnet | AutoAttack | VGG
LABELS_RESNET_AUTOATTACK_VGG = PROJECT_DIR + 'prediction_results/ResNet_AutoAttack_VGG.json'
IMAGES_RESNET_AUTOATTACK_VGG_DIR = PROJECT_DIR + 'test_ResNet_AutoAttack'

print(IMAGES_RESNET_AUTOATTACK_VGG_DIR)
adv_resnet_autoattack_vgg_dataset = AdversarialImageDataset(root_dir=IMAGES_RESNET_AUTOATTACK_VGG_DIR, transform=preprocess)
adv_resnet_autoattack_vgg_loader = DataLoader(adv_resnet_autoattack_vgg_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_OnePixel
/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_PGD
/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_FGSM
/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_DeepFool
/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_CW
/content/drive/Shareddrives/adv_images/adversarial_images/test_ResNet_AutoAttack


In [16]:
len(adv_resnet_autoattack_vgg_dataset)

10000

In [17]:
with open(LABELS_CLEAN_VGG, 'r') as f:
    labels_clean_vgg = json.load(f)
    labels_clean_vgg = labels_clean_vgg['clean_VGG']

with open(LABELS_RESNET_ONEPIXEL_VGG, 'r') as f:
    labels_resnet_onepixel_vgg = json.load(f)
    labels_resnet_onepixel_vgg = labels_resnet_onepixel_vgg['ResNet_OnePixel_VGG']

with open(LABELS_RESNET_PGD_VGG, 'r') as f:
    labels_resnet_pgd_vgg = json.load(f)
    labels_resnet_pgd_vgg = labels_resnet_pgd_vgg['ResNet_PGD_VGG']

with open(LABELS_RESNET_FGSM_VGG, 'r') as f:
    labels_resnet_fgsm_vgg = json.load(f)
    labels_resnet_fgsm_vgg = labels_resnet_fgsm_vgg['ResNet_FGSM_VGG']

with open(LABELS_RESNET_DEEPFOOL_VGG, 'r') as f:
    labels_resnet_deepfool_vgg = json.load(f)
    labels_resnet_deepfool_vgg = labels_resnet_deepfool_vgg['ResNet_DeepFool_VGG']

with open(LABELS_RESNET_CW_VGG, 'r') as f:
    labels_resnet_cw_vgg = json.load(f)
    labels_resnet_cw_vgg = labels_resnet_cw_vgg['ResNet_CW_VGG']

with open(LABELS_RESNET_AUTOATTACK_VGG, 'r') as f:
    labels_resnet_autoattack_vgg = json.load(f)
    labels_resnet_autoattack_vgg = labels_resnet_autoattack_vgg['ResNet_AutoAttack_VGG']

metadata_resnet_onepixel_vgg = AttackMetadata("OnePixel", "ResNet", "VGG")
metadata_resnet_pgd_vgg = AttackMetadata("PGD", "ResNet", "VGG")
metadata_resnet_fgsm_vgg = AttackMetadata("FGSM", "ResNet", "VGG")
metadata_resnet_deepfool_vgg = AttackMetadata("DeepFool", "ResNet", "VGG")
metadata_resnet_cw_vgg = AttackMetadata("CW", "ResNet", "VGG")
metadata_resnet_autoattack_vgg = AttackMetadata("AutoAttack", "ResNet", "VGG")

In [18]:
imageinfos_resnet_onepixel_vgg = []
imageinfos_resnet_pgd_vgg = []
imageinfos_resnet_fgsm_vgg = []
imageinfos_resnet_deepfool_vgg = []
imageinfos_resnet_cw_vgg = []
imageinfos_resnet_autoattack_vgg = []

for idx, filename in enumerate(adv_resnet_onepixel_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))

      # attak_metadata, index, label, json clean label, json attack label, (featuremap clean), (featuremap attacked)
      imageinfos_resnet_onepixel_vgg.append(ImageInfo(metadata_resnet_onepixel_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_onepixel_vgg[idx]))

for idx, filename in enumerate(adv_resnet_pgd_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))
      imageinfos_resnet_pgd_vgg.append(ImageInfo(metadata_resnet_pgd_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_pgd_vgg[idx]))

for idx, filename in enumerate(adv_resnet_fgsm_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))

      imageinfos_resnet_fgsm_vgg.append(ImageInfo(metadata_resnet_fgsm_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_fgsm_vgg[idx]))

for idx, filename in enumerate(adv_resnet_deepfool_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))

      imageinfos_resnet_deepfool_vgg.append(ImageInfo(metadata_resnet_deepfool_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_deepfool_vgg[idx]))

for idx, filename in enumerate(adv_resnet_cw_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))

      imageinfos_resnet_cw_vgg.append(ImageInfo(metadata_resnet_cw_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_cw_vgg[idx]))

for idx, filename in enumerate(adv_resnet_autoattack_vgg_dataset.image_files):
      match = re.match(r"adv_image_(\d+)_label_(\d+)\.png", filename)
      image_index, true_label = int(match.group(1)), int(match.group(2))

      imageinfos_resnet_autoattack_vgg.append(ImageInfo(metadata_resnet_autoattack_vgg, image_index, true_label,
                                                        labels_clean_vgg[idx], labels_resnet_autoattack_vgg[idx]))

print(f"Processed {len(imageinfos_resnet_onepixel_vgg)}, {len(imageinfos_resnet_pgd_vgg)} and created ImageInfo objects.")

Processed 10000, 10000 and created ImageInfo objects.


In [19]:
num_examples_to_print = 10
print(f"\nFirst {num_examples_to_print} image prediction infos (calling .to_dict()):")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_onepixel_vgg[i].to_dict())
print(20*"=")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_pgd_vgg[i].to_dict())
print(20*"=")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_fgsm_vgg[i].to_dict())
print(20*"=")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_deepfool_vgg[i].to_dict())
print(20*"=")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_cw_vgg[i].to_dict())
print(20*"=")
for i in range(num_examples_to_print):
    print(imageinfos_resnet_autoattack_vgg[i].to_dict())


First 10 image prediction infos (calling .to_dict()):
{'image_index': 0, 'true_label': 3, 'clean_prediction_label': 3, 'adversarial_prediction_label': 3, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': True, 'was_attack_successful': False, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_index': 1, 'true_label': 8, 'clean_prediction_label': 8, 'adversarial_prediction_label': 5, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': False, 'was_attack_successful': True, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_index': 2, 'true_label': 8, 'clean_prediction_label': 8, 'adversarial_prediction_label': 8, 'is_clean_prediction_correct': True, 'is_adversarial_prediction_correct': True, 'was_attack_successful': False, 'attack_metadata': {'attack_name': 'OnePixel', 'generated_by': 'ResNet', 'model_attacked': 'VGG'}}
{'image_inde

In [21]:
model = InceptionCIFAR(num_classes=10)
model.load_state_dict(torch.load("/content/drive/MyDrive/TAI/sem2/DL/inception_128/251216_Inception_epoch100/model_Inception_cifar10_251216.pth", map_location=device))
model.to(device)
model.eval()

InceptionCIFAR(
  (stem): Sequential(
    (0): Conv2d(3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (inception3a): InceptionModule(
    (b1): Sequential(
      (0): Conv2d(96, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
    )
    (b2): Sequential(
      (0): Conv2d(96, 48, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (1): BatchNorm2d(48, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): Conv2d(48, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (4): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (5): ReLU(inplace=True)
    )
    (b3): Sequential(
      (0): Conv2d(96, 8, kernel_size=(1, 1), 

In [26]:
CLEAN_VGG_FM = FEATURE_EXTRACTION_RESULTS + "clean_VGG"
RESNET_ONEPIXEL_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_OnePixel_VGG"
RESNET_PGD_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_PGD_VGG"
RESNET_FGSM_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_FGSM_VGG"
RESNET_DEEPFOOL_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_DeepFool_VGG"
RESNET_CW_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_CW_VGG"
RESNET_AUTOATTACK_VGG_FM = FEATURE_EXTRACTION_RESULTS + "ResNet_AutoAttack_VGG"

In [25]:
def extract_clean_features(loader, save_dir):
    global_idx = 0

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Clean Batches"):
            images = images.to(device)
            features = model.forward_features(images)
            features_np = features.cpu().numpy()

            for i in range(features_np.shape[0]):
                np.save(os.path.join(save_dir, f"clean_feature_{global_idx}.npy"), features_np[i])
                global_idx += 1

def extract_adv_features(loader, image_infos, save_dir):
    print(f"--- Przetwarzanie Ataku -> {save_dir} ---")

    current_idx = 0

    with torch.no_grad():
        for images, _ in tqdm(loader, desc="Adv Batches"):
            images = images.to(device)
            features = model.forward_features(images)
            features_np = features.cpu().numpy()

            for i in range(features_np.shape[0]):
                info = image_infos[current_idx]
                original_id = info.image_index

                np.save(os.path.join(save_dir, f"adv_feature_{original_id}.npy"), features_np[i])
                info.adversarial_feature_map = features_np[i]

                current_idx += 1

extract_clean_features(clean_loader, CLEAN_VGG_FM) # TODO: zapisz w każdym image info

extract_adv_features(
    loader=adv_resnet_onepixel_vgg_loader,
    image_infos=imageinfos_resnet_onepixel_vgg,
    save_dir=RESNET_ONEPIXEL_VGG_FM
)

Clean Batches:   0%|          | 0/79 [00:00<?, ?it/s]

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_OnePixel_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

In [27]:
# pgd
extract_adv_features(
    loader=adv_resnet_pgd_vgg_loader,
    image_infos=imageinfos_resnet_pgd_vgg,
    save_dir=RESNET_PGD_VGG_FM
)
#fgsm
extract_adv_features(
    loader=adv_resnet_fgsm_vgg_loader,
    image_infos=imageinfos_resnet_fgsm_vgg,
    save_dir=RESNET_FGSM_VGG_FM
)
# deep fool
extract_adv_features(
    loader=adv_resnet_deepfool_vgg_loader,
    image_infos=imageinfos_resnet_deepfool_vgg,
    save_dir=RESNET_DEEPFOOL_VGG_FM
)
# cw
extract_adv_features(
    loader=adv_resnet_cw_vgg_loader,
    image_infos=imageinfos_resnet_cw_vgg,
    save_dir=RESNET_CW_VGG_FM
)
# auto attack
extract_adv_features(
    loader=adv_resnet_autoattack_vgg_loader,
    image_infos=imageinfos_resnet_autoattack_vgg,
    save_dir=RESNET_AUTOATTACK_VGG_FM
)

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_PGD_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_FGSM_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_DeepFool_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_CW_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

--- Przetwarzanie Ataku -> /content/drive/Shareddrives/adv_images/adversarial_images/feature_extraction/ResNet_AutoAttack_VGG ---


Adv Batches:   0%|          | 0/79 [00:00<?, ?it/s]

## TODO:
- wyznaczenie normy L2​ i Cosine Similarity dla każdego rodzaju ataku
  - porówanie wyników między różnymi atakami
  - porównanie wyników między klasami CIFAR-10
- sprawdzenie, jak często atak wygenerowany dla klasy A powoduje błędną klasyfikację jako klasa B i czy ma to związek z mapami cech